# wav2vec2 Audio Processing Pipeline (SQL-backed)

Reproducible **processing** for the wav2vec2 speech-emotion model (EDA + training live in their own notebooks).
Mirrors `sqltransfer.ipynb`, but matches what wav2vec2 needs instead of the MFCC / log-mel pipeline:

- **No log-mel features** — wav2vec2 learns from the raw 16 kHz waveform, so the only audio step is a
  resample to 16 kHz mono (capped at 5 s, no padding — the model's collator pads per batch).
- **Duration filter 1–5 s** (not pad-to-3s) — same window as the wav2vec2 notebook.
- **Speaker-INDEPENDENT 3-way split** (train / val / test) via `GroupShuffleSplit(test_size=0.15,
  random_state=42)`, all speaker-disjoint — identical params to the wav2vec2 training notebook, so the
  test clips stay the same and accuracy is comparable to the ~0.63 MFCC baseline and the 0.705 wav2vec2 number.

Steps: seeds → connect → metadata → filter → speaker split → resample → **push cleaned dataset (last)** → repro check.

> If the DB is unreachable, `mysql_ok` stays False and everything still runs in memory — the pipeline never blocks.

In [ ]:
# === Imports + fixed random seeds (run first, every session) ===
import os, re, hashlib, random
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.auto import tqdm

# Load DB credentials from the repo-root .env (gitignored) so the kernel sees MYSQL_* without
# needing shell `export`s. override=True so .env wins over any stale shell exports.
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(), override=True)
except ImportError:
    pass  # pip install python-dotenv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    _t = "/torch"
except ImportError:
    _t = ""
print(f"seeds set to {SEED}  (numpy/random{_t})")

In [ ]:
# === Config ===
EMO_DIR       = Path("../Emotions")              # raw dataset root: <emotion>/*.wav
PROCESSED_DIR = Path("../Emotions_w2v2_16k")     # 16 kHz output (NEVER write into EMO_DIR)
SR            = 16000                             # wav2vec2 sampling rate
DUR_MIN, DUR_MAX = 1.0, 5.0                       # keep clips in this duration window (sec)
MAX_SEC       = 5.0                               # cap clip length (no padding; collator pads per batch)

# Cleaned-dataset scope: reduce to these sources, drop these (misspelled / out-of-scope) emotions.
KEEP_SOURCES = ("CREMA-D", "RAVDESS")
DROP_LABELS  = ("Suprised",)

def source_of(name):
    if re.match(r"^\d+-\d+-", name):             return "RAVDESS"
    if re.match(r"^\d{4}_", name):               return "CREMA-D"
    if re.match(r"^(OAF|YAF|OA)_", name):        return "TESS"
    if re.match(r"^[a-z]{1,2}\d+\.wav$", name):  return "SAVEE"
    return "other"

def speaker_of(filename, source):
    # speaker id for GroupShuffleSplit -- same logic as the wav2vec2 notebook
    if source == "CREMA-D":   return f"CREMA-{filename[:4]}"
    if source == "RAVDESS":   return f"RAVDESS-{filename.split('.')[0].split('-')[-1]}"
    return f"{source}-{filename}"

In [ ]:
# === Database connection (cloud MySQL, e.g. Aiven) — built ONCE, reused by every DB cell below ===
# Credentials come from .env / shell env (see .env.example). Cloud MySQL enforces TLS, so we always
# pass an SSL arg. If the DB is unreachable, mysql_ok stays False and the pipeline falls back to the
# in-memory table so it never blocks.
from sqlalchemy import create_engine, text

USER = os.getenv("MYSQL_USER", "root")
PWD  = os.getenv("MYSQL_PASSWORD", "")
HOST = os.getenv("MYSQL_HOST", "localhost")
PORT = os.getenv("MYSQL_PORT", "3306")
DB   = os.getenv("MYSQL_DB",   "ser")
SSL_CA = os.getenv("MYSQL_SSL_CA", "")           # path to ca.pem for strict CA verification (optional)
CONNECT_ARGS = {"ssl": {"ca": SSL_CA}} if SSL_CA else {"ssl": {"ssl": True}}

def recreate_table(name, ddl):
    """DROP + CREATE a table with an explicit PK (Aiven sets sql_require_primary_key=ON)."""
    with engine.begin() as conn:
        conn.execute(text(f"DROP TABLE IF EXISTS {name}"))
        conn.execute(text(ddl))

mysql_ok = False
try:
    engine = create_engine(f"mysql+pymysql://{USER}:{PWD}@{HOST}:{PORT}/{DB}", connect_args=CONNECT_ARGS)
    with engine.connect() as conn:
        v = conn.execute(text("SELECT VERSION()")).scalar()
    mysql_ok = True
    print(f"OK  connected to {HOST}:{PORT}/{DB}  (MySQL {v})")
except Exception as e:
    print(f"NOT connected ({type(e).__name__}: {e})")
    print("check: .env values set? service Running? port is the 5-digit Aiven port, not 3306?")
    print("-> pipeline will continue with local (in-memory) filtering.")

In [ ]:
# === Build the metadata table (one row per file) ===
# Header reads only (soundfile.info) -> fast. Flags corrupted (unreadable / 0 frames) and duplicate
# (identical bytes). Nothing deleted -- flags only; md5 is dropped after the dup check.
def file_md5(p, chunk=1 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

rows = []
for emo_path in sorted(p for p in EMO_DIR.iterdir() if p.is_dir()):
    for wav in sorted(emo_path.glob("*.wav")):
        rec = {"filename": wav.name, "file_path": str(wav), "emotion": emo_path.name,
               "source": source_of(wav.name), "file_size": wav.stat().st_size}
        try:
            info = sf.info(str(wav))
            rec["duration_sec"] = round(info.frames / info.samplerate, 3)
            rec["sample_rate"]  = int(info.samplerate)
            rec["channels"]     = int(info.channels)
            rec["corrupted_flagged"] = bool(info.frames == 0)
        except Exception:
            rec.update(duration_sec=None, sample_rate=None, channels=None, corrupted_flagged=True)
        rec["md5"] = file_md5(wav) if not rec["corrupted_flagged"] else None
        rows.append(rec)

meta = pd.DataFrame(rows)
meta["duplicate_flagged"] = meta["md5"].duplicated(keep="first") & meta["md5"].notna()
meta = meta.drop(columns=["md5"])               # hash only needed to compute duplicate_flagged
meta.insert(0, "clip_id", range(1, len(meta) + 1))
print(f"{len(meta)} files | corrupted: {int(meta['corrupted_flagged'].sum())} | "
      f"duplicates: {int(meta['duplicate_flagged'].sum())}")
meta.head()

In [ ]:
# === Filter to the CLEAN working set (in memory) ===
# Clean = not corrupted, not duplicate, CREMA-D + RAVDESS only, 'Suprised' dropped, duration 1-5 s
# (same duration window as the wav2vec2 notebook).
clean = meta[(~meta["corrupted_flagged"])
             & (~meta["duplicate_flagged"])
             & (meta["source"].isin(KEEP_SOURCES))
             & (~meta["emotion"].isin(DROP_LABELS))
             & (meta["duration_sec"] >= DUR_MIN)
             & (meta["duration_sec"] <= DUR_MAX)].reset_index(drop=True)
print(f"clean clips: {len(clean)} / {len(meta)} total  "
      f"({', '.join(KEEP_SOURCES)}, no {'/'.join(DROP_LABELS)}, {DUR_MIN}-{DUR_MAX}s)")
print(clean["emotion"].value_counts())

In [ ]:
# === Speaker-INDEPENDENT 3-way split (train / val / test, all speaker-disjoint) ===
# Identical params to the wav2vec2 notebook: GroupShuffleSplit(test_size=0.15, random_state=42),
# applied twice -- hold out test, then carve val out of the train pool. Same seed -> same test clips.
from sklearn.model_selection import GroupShuffleSplit

clean = clean.copy()
clean["speaker"] = [speaker_of(n, s) for n, s in zip(clean["filename"], clean["source"])]

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
trv_idx, te_idx = next(gss.split(clean, clean["emotion"], groups=clean["speaker"]))
trv  = clean.iloc[trv_idx]
test = clean.iloc[te_idx]

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr_rel, va_rel = next(gss_val.split(trv, trv["emotion"], groups=trv["speaker"]))
val = trv.iloc[va_rel]

clean["split"] = "train"                         # default
clean.loc[val.index,  "split"] = "val"
clean.loc[test.index, "split"] = "test"

# guard: no speaker may appear in more than one split
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    sa = set(clean.loc[clean["split"] == a, "speaker"])
    sb = set(clean.loc[clean["split"] == b, "speaker"])
    assert not (sa & sb), f"speaker leakage between {a}/{b}"
print(clean["split"].value_counts())

In [ ]:
# === Resample each clean clip to 16 kHz mono (<=5 s) -> PROCESSED_DIR (originals untouched) ===
# wav2vec2 wants 16 kHz; doing it once here means training doesn't resample every epoch. Variable
# length is fine (the model's collator pads per batch), so we cap at MAX_SEC but do NOT pad.
MAX_LEN = int(MAX_SEC * SR)

def resample_file(src_path, emotion):
    out_dir = PROCESSED_DIR / emotion
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / Path(src_path).name
    if out_path.exists():
        return str(out_path)                       # idempotent: skip already-processed
    y, _ = librosa.load(src_path, sr=SR, mono=True)
    y = y[:MAX_LEN]                                 # cap at 5 s, no padding
    sf.write(str(out_path), y, SR)
    return str(out_path)

clean["processed_path"] = [resample_file(p, e) for p, e in
                           tqdm(list(zip(clean["file_path"], clean["emotion"])), desc="resample")]
print("16 kHz audio ->", PROCESSED_DIR.resolve())

In [ ]:
# === Push the CLEANED dataset to MySQL (after all processing) ===
# Single source of truth for the wav2vec2 model: one row per clean clip with its 16 kHz path,
# speaker, and 3-way split. PK up front (Aiven requires it), then rows appended.
DATASET_TABLE = "audio_dataset_w2v2"
DATASET_DDL = f"""
CREATE TABLE {DATASET_TABLE} (
    clip_id        INTEGER      NOT NULL PRIMARY KEY,
    filename       VARCHAR(256),
    file_path      VARCHAR(512),
    processed_path VARCHAR(512),
    emotion        VARCHAR(32),
    source         VARCHAR(16),
    duration_sec   FLOAT,
    speaker        VARCHAR(64),
    split          VARCHAR(8),
    INDEX idx_{DATASET_TABLE}_emotion (emotion),
    INDEX idx_{DATASET_TABLE}_split (split),
    INDEX idx_{DATASET_TABLE}_speaker (speaker)
)
"""
COLS = ["clip_id", "filename", "file_path", "processed_path", "emotion",
        "source", "duration_sec", "speaker", "split"]

if mysql_ok:
    recreate_table(DATASET_TABLE, DATASET_DDL)
    clean[COLS].to_sql(DATASET_TABLE, engine, if_exists="append", index=False)
    with engine.connect() as conn:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {DATASET_TABLE}")).scalar()
    print(f"wrote {n} rows to MySQL `{DB}.{DATASET_TABLE}`  (cleaned + 3-way speaker-disjoint split)")
else:
    print("MySQL not connected -> push skipped; cleaned dataset lives in the `clean` DataFrame")

In [ ]:
# === Reproducibility check (per-split counts + speaker-disjoint guarantee) ===
print("SEED:", SEED)
print("\nclips per split:");    print(clean["split"].value_counts())
print("\nspeakers per split:"); print(clean.groupby("split")["speaker"].nunique())
print("\nemotion x split:");    print(pd.crosstab(clean["emotion"], clean["split"]))

## Reproducibility & sharing

- This notebook is the shared **wav2vec2** preprocessing script — commit it so the team runs the same file.
- `SEED = 42` + `GroupShuffleSplit(test_size=0.15, random_state=42)` match the wav2vec2 training notebook,
  so the test clips are the SAME speaker-disjoint set and accuracy stays comparable across runs.
- The MySQL **`audio_dataset_w2v2`** table is the single source of truth. The training notebook reads:
  ```sql
  SELECT processed_path, emotion FROM audio_dataset_w2v2 WHERE split = 'train';
  SELECT processed_path, emotion FROM audio_dataset_w2v2 WHERE split = 'val';
  SELECT processed_path, emotion FROM audio_dataset_w2v2 WHERE split = 'test';
  ```
- Credentials live in the same local `.env` (gitignored). Originals in `Emotions/` are never modified;
  16 kHz clips are written to `Emotions_w2v2_16k/`.